# BANELO SALES FORECASTING - ML MODEL TRAINING
## Linear Regression vs Gradient Boosting Comparison

This notebook trains two machine learning models for sales forecasting:
1. **Linear Regression** (Baseline Model)
2. **Gradient Boosting Regressor** (Main Model)

Models are evaluated using MAE, MAPE, and RMSE metrics.

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ All dependencies loaded!')

## Upload Sales Data

Click the folder icon on the left, then upload your `sales_for_colab.csv` file

In [ ]:
from google.colab import files

print('📁 Upload your sales_for_colab.csv file...')
uploaded = files.upload()

if 'sales_for_colab.csv' in uploaded:
    df = pd.read_csv('sales_for_colab.csv')
    print('✅ File uploaded successfully!')
else:
    print('❌ Expected file not found')

## Data Exploration

In [ ]:
print('=' * 70)
print('DATA OVERVIEW')
print('=' * 70)

print(f'\n📊 Dataset Shape: {df.shape[0]} records')
print(f'\n📅 Date Range: {df["date"].min()} to {df["date"].max()}')
print(f'\n🏷️  Categories: {df["category"].unique().tolist()}')
print(f'📦 Products: {df["product_name"].nunique()}')
print(f'\n📝 Sample Data:')
print(df.head(10))

## Data Preparation

In [ ]:
df['date'] = pd.to_datetime(df['date'])

daily_sales = df.groupby(['date', 'product_name', 'category']).agg({
    'quantity': 'sum',
    'total': 'sum',
    'price': 'mean'
}).reset_index().sort_values('date')

print(f'✅ Created {len(daily_sales)} daily records')
print(f'   Unique products: {daily_sales["product_name"].nunique()}')

## Feature Engineering

In [ ]:
print('\n' + '=' * 70)
print('FEATURE ENGINEERING')
print('=' * 70)

df_features = daily_sales.copy()

# TIME-BASED FEATURES
print('\n⏰ Creating time-based features...')
df_features['day_of_week'] = df_features['date'].dt.dayofweek
df_features['day_of_month'] = df_features['date'].dt.day
df_features['month'] = df_features['date'].dt.month
df_features['year'] = df_features['date'].dt.year
df_features['week_of_year'] = df_features['date'].dt.isocalendar().week
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)
df_features['days_since_start'] = (df_features['date'] - df_features['date'].min()).dt.days

# CATEGORY ENCODING
print('\n🏷️  Encoding categorical features...')
category_encoder = LabelEncoder()
df_features['category_encoded'] = category_encoder.fit_transform(df_features['category'])

# LAG FEATURES
print('\n📉 Creating lag features...')
for lag in [1, 7, 14]:
    df_features[f'quantity_lag_{lag}'] = df_features.groupby('product_name')['quantity'].shift(lag)

# ROLLING STATISTICS
print('\n📊 Creating rolling statistics...')
for window in [7, 14, 30]:
    df_features[f'quantity_ma_{window}'] = df_features.groupby('product_name')['quantity'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

df_features = df_features.fillna(0)

feature_columns = [
    'day_of_week', 'day_of_month', 'month', 'year', 'week_of_year',
    'is_weekend', 'days_since_start', 'category_encoded',
    'quantity_lag_1', 'quantity_lag_7', 'quantity_lag_14',
    'quantity_ma_7', 'quantity_ma_14', 'quantity_ma_30'
]

print(f'\n✅ Total features: {len(feature_columns)}')

## Model Training

In [ ]:
print('\n' + '=' * 70)
print('MODEL TRAINING')
print('=' * 70)

X = df_features[feature_columns]
y = df_features['quantity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'\n📊 Data Split:')
print(f'   Training: {X_train.shape[0]} samples')
print(f'   Testing: {X_test.shape[0]} samples')

### Linear Regression (Baseline)

In [ ]:
print('\n📈 TRAINING LINEAR REGRESSION...')

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_test_pred = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_test_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_test_pred))
lr_mape = mean_absolute_percentage_error(y_test, np.maximum(lr_test_pred, 0.01))
lr_r2 = r2_score(y_test, lr_test_pred)

print(f'\n✅ Linear Regression Metrics:')
print(f'   MAE:  {lr_mae:.2f}')
print(f'   RMSE: {lr_rmse:.2f}')
print(f'   MAPE: {lr_mape:.2%}')
print(f'   R²:   {lr_r2:.4f}')

lr_metrics = {'MAE': lr_mae, 'RMSE': lr_rmse, 'MAPE': lr_mape, 'R2': lr_r2}

### Gradient Boosting (Main Model)

In [ ]:
print('\n🚀 TRAINING GRADIENT BOOSTING...')

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42
)

gb_model.fit(X_train, y_train)
gb_test_pred = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_test_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_test_pred))
gb_mape = mean_absolute_percentage_error(y_test, np.maximum(gb_test_pred, 0.01))
gb_r2 = r2_score(y_test, gb_test_pred)

print(f'\n✅ Gradient Boosting Metrics:')
print(f'   MAE:  {gb_mae:.2f}')
print(f'   RMSE: {gb_rmse:.2f}')
print(f'   MAPE: {gb_mape:.2%}')
print(f'   R²:   {gb_r2:.4f}')

gb_metrics = {'MAE': gb_mae, 'RMSE': gb_rmse, 'MAPE': gb_mape, 'R2': gb_r2}

### Model Comparison

In [ ]:
print('\n' + '=' * 70)
print('MODEL COMPARISON')
print('=' * 70)

comparison_df = pd.DataFrame({
    'Linear Regression': [f'{lr_mae:.2f}', f'{lr_rmse:.2f}', f'{lr_mape:.2%}', f'{lr_r2:.4f}'],
    'Gradient Boosting': [f'{gb_mae:.2f}', f'{gb_rmse:.2f}', f'{gb_mape:.2%}', f'{gb_r2:.4f}']
}, index=['MAE', 'RMSE', 'MAPE', 'R²'])

print('\n📊 Performance Metrics:')
print(comparison_df.to_string())

print('\n🏆 Winner Analysis:')
print(f'   MAE:  Gradient Boosting' if gb_mae < lr_mae else f'   MAE:  Linear Regression')
print(f'   RMSE: Gradient Boosting' if gb_rmse < lr_rmse else f'   RMSE: Linear Regression')
print(f'   MAPE: Gradient Boosting' if gb_mape < lr_mape else f'   MAPE: Linear Regression')
print(f'   R²:   Gradient Boosting' if gb_r2 > lr_r2 else f'   R²:   Linear Regression')

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\n🚀 Top 10 Important Features:')
print(feature_importance.head(10).to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
top_features = feature_importance.head(10)
ax.barh(top_features['feature'], top_features['importance'], color='steelblue')
ax.set_xlabel('Importance Score')
ax.set_title('Top 10 Feature Importance - Gradient Boosting Model')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Feature importance chart saved!')

## Generate Forecasts

In [ ]:
print('\n' + '=' * 70)
print('GENERATING 30-DAY FORECAST')
print('=' * 70)

last_date = df_features['date'].max()
forecast_dates = pd.date_range(start=last_date + timedelta(days=1), periods=30, freq='D')

print(f'\n📅 Forecast Period: {forecast_dates[0].date()} to {forecast_dates[-1].date()}')

daily_forecasts = []
last_row = df_features.iloc[-1][feature_columns].values.reshape(1, -1)

for forecast_date in forecast_dates:
    forecast_features = last_row.copy()
    forecast_features[0][0] = forecast_date.dayofweek
    forecast_features[0][1] = forecast_date.day
    forecast_features[0][2] = forecast_date.month
    forecast_features[0][3] = forecast_date.year
    forecast_features[0][4] = forecast_date.isocalendar()[1]
    forecast_features[0][5] = 1 if forecast_date.dayofweek in [5, 6] else 0
    
    pred = gb_model.predict(forecast_features)[0]
    daily_forecasts.append({
        'date': forecast_date.date(),
        'forecast_quantity': max(0, pred)
    })

forecast_df = pd.DataFrame(daily_forecasts)

print(f'\n📊 Daily Forecast (First 10 days):')
print(forecast_df.head(10).to_string(index=False))

monthly_total = forecast_df['forecast_quantity'].sum()
print(f'\n📈 30-Day Total Forecast: {monthly_total:.2f} units')

## Generate Report

In [ ]:
print('\n' + '=' * 70)
print('GENERATING PROFESSIONAL REPORT')
print('=' * 70)

report_content = f"""{'=' * 80}
BANELO SALES FORECASTING - GRADIENT BOOSTING REPORT
{'=' * 80}

📅 TRAINING INFORMATION
{'-' * 80}
Training Period: {df_features['date'].min().date()} to {df_features['date'].max().date()}
Total Records: {len(df_features)}
Unique Products: {df_features['product_name'].nunique()}
Number of Features: {len(feature_columns)}

{'=' * 80}
MODEL PERFORMANCE
{'=' * 80}

Linear Regression (Baseline Model):
   MAE (Mean Absolute Error): {lr_mae:.2f}
   RMSE (Root Mean Squared Error): {lr_rmse:.2f}
   MAPE (Mean Absolute Percentage Error): {lr_mape:.2%}
   R² Score: {lr_r2:.4f}

Gradient Boosting (Main Model):
   MAE (Mean Absolute Error): {gb_mae:.2f}
   RMSE (Root Mean Squared Error): {gb_rmse:.2f}
   MAPE (Mean Absolute Percentage Error): {gb_mape:.2%}
   R² Score: {gb_r2:.4f}

🏆 Best Model: Gradient Boosting

{'=' * 80}
TOP 10 IMPORTANT FEATURES
{'=' * 80}

"""

for idx, (_, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    report_content += f"{idx:2d}. {row['feature']:25s} - {row['importance']:.4f}\n"

report_content += f"""\n{'=' * 80}
30-DAY FORECAST
{'=' * 80}

Forecast Period: {forecast_dates[0].date()} to {forecast_dates[-1].date()}
Total 30-Day Forecast: {monthly_total:.2f} units
Average Daily Forecast: {monthly_total/30:.2f} units

{'=' * 80}
KEY INSIGHTS
{'=' * 80}

✓ Gradient Boosting Advantages:
  - Handles non-linear relationships in sales data
  - Better performance than baseline Linear Regression model
  - Feature importance shows key business drivers
  - Superior prediction accuracy

{'=' * 80}
METRICS EXPLAINED
{'=' * 80}

MAE (Mean Absolute Error):
  - Average absolute difference between predicted and actual sales
  - Interpretation: On average, prediction was off by {gb_mae:.2f} units
  - Lower is better

RMSE (Root Mean Squared Error):
  - Penalizes large prediction errors more heavily
  - Lower is better

MAPE (Mean Absolute Percentage Error):
  - Measures prediction error as a percentage
  - Interpretation: Prediction was off by {gb_mape:.2%} on average
  - Lower is better

R² Score:
  - Proportion of variance explained by the model
  - Interpretation: Model explains {gb_r2*100:.2f}% of the variance
  - Higher is better (0-1 scale)

{'=' * 80}

Report generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

{'=' * 80}
"""

with open('Banelo_Sales_Forecasting_Report.txt', 'w') as f:
    f.write(report_content)

print('\n✅ Report saved: Banelo_Sales_Forecasting_Report.txt')
print(report_content)

## Save Models & Forecasts

In [ ]:
print('\n' + '=' * 70)
print('SAVING MODELS')
print('=' * 70)

joblib.dump(gb_model, 'gradient_boosting_model.pkl')
joblib.dump(lr_model, 'linear_regression_model.pkl')
joblib.dump(feature_columns, 'feature_columns.pkl')
joblib.dump(category_encoder, 'category_encoder.pkl')
forecast_df.to_csv('forecasts_30day.csv', index=False)

print('\n✅ Models saved successfully!')
print('\nFiles ready for download:')
print('   1. gradient_boosting_model.pkl')
print('   2. linear_regression_model.pkl')
print('   3. feature_columns.pkl')
print('   4. category_encoder.pkl')
print('   5. forecasts_30day.csv')
print('   6. Banelo_Sales_Forecasting_Report.txt')
print('   7. feature_importance.png')

## Summary

In [ ]:
print('\n' + '=' * 70)
print('TRAINING COMPLETE! 🎉')
print('=' * 70)

print(f"""
✅ SUMMARY:
   Linear Regression MAE: {lr_mae:.2f}
   Gradient Boosting MAE: {gb_mae:.2f}
   Improvement: {((lr_mae - gb_mae) / lr_mae * 100):.1f}%
   
   Total features: {len(feature_columns)}
   Training records: {len(df_features)}
   30-day forecast: {monthly_total:.2f} units

🎯 NEXT STEPS:
   1. Download all .pkl files
   2. Download Banelo_Sales_Forecasting_Report.txt
   3. Download forecasts_30day.csv
   4. Move .pkl files to: ml_models/ folder
   5. Run: python integrate_ml_model.py

📚 THESIS DELIVERABLES:
   ✓ Model comparison (LR vs GB)
   ✓ Performance metrics (MAE, RMSE, MAPE, R²)
   ✓ Feature importance analysis
   ✓ Professional research report
""")